# Notebook 01 — Data Collection & Enrichment

**Goal:** Randomly sample 10,000 Last.fm users via the API, fetch their 5-year
scrobble histories, and enrich with Spotify genre and audio feature data.

**How users are discovered — tag → artist → fan graph**
1. Fetch the top global Last.fm tags (genres / moods).
2. For each tag, pull the top artists.
3. For each artist, pull their top fans (real Last.fm usernames).
4. Deduplicate the union and randomly sample 10,000 users.

This produces a diverse, genre-balanced cohort without needing the 2.53 GB
baseline TSV file.

**Inputs:**
- Last.fm API credentials in `.env`
- Spotify API credentials in `.env`

**Outputs (all in `data/processed/`, all CSV):**
- `scrobbles_updated.csv` — fetched scrobble history for sampled users
- `profiles.csv`          — stub user demographics (filled from API where available)
- `artist_genres.csv`     — Spotify genre lookup table
- `audio_features.csv`    — Spotify audio features for top tracks

> **Note:** This notebook only needs to run once. All subsequent notebooks
> read from the saved CSV files.  Re-running is safe — per-user scrobble
> caches in `data/processed/lastfm_user_cache/` let the fetch resume where
> it left off.

In [ ]:
import logging
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)

Path('../data/processed').mkdir(parents=True, exist_ok=True)
Path('../data/processed/lastfm_user_cache').mkdir(parents=True, exist_ok=True)
Path('../outputs/figures').mkdir(parents=True, exist_ok=True)

## 1. Verify environment and credentials

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv('../.env')

required_vars = ['LASTFM_API_KEY', 'LASTFM_API_SECRET', 'SPOTIFY_CLIENT_ID', 'SPOTIFY_CLIENT_SECRET']
for var in required_vars:
    val = os.environ.get(var, '')
    status = '✓' if val else '✗ MISSING'
    print(f'  {var}: {status}')

## 2. Load config & initialise APIs

In [ ]:
from src.data.loader import load_config
from src.data.lastfm_client import build_network

cfg = load_config('../configs/config.yaml')
dc  = cfg['data_collection']

network = build_network()
print('Last.fm network initialised.')
print(f'Target sample size : {dc["sample_users"]:,} users')
print(f'Discovery tags     : {dc["discovery_n_tags"]}')
print(f'Artists per tag    : {dc["discovery_n_artists_per_tag"]}')
print(f'Fans per artist    : {dc["discovery_n_fans_per_artist"]}')
print(f'Lookback window    : {dc["lookback_years"]} years')

## 3. Discover random users via tag → artist → fan graph

In [ ]:
from src.data.lastfm_client import discover_random_users

# This cell makes API calls but results are deterministic (fixed seed).
# Re-run is fast because the user list is reproducible from the same seed.
sampled_userids = discover_random_users(
    network=network,
    target_n=dc['sample_users'],
    n_tags=dc['discovery_n_tags'],
    n_artists_per_tag=dc['discovery_n_artists_per_tag'],
    n_fans_per_artist=dc['discovery_n_fans_per_artist'],
    seed=dc['discovery_seed'],
    request_delay=dc['lastfm_request_delay'],
)

print(f'Users discovered: {len(sampled_userids):,}')
print(f'Sample (first 10): {sampled_userids[:10]}')

In [ ]:
# Create a profiles stub CSV — no demographic TSV needed.
# If you have the original userid-profile.tsv you can load it with
#   load_profiles('../data/raw/userid-profile.tsv', save_csv='../data/processed/profiles.csv')
# and replace `sampled_userids` below with the profile-derived user list.

import pandas as pd

profiles = pd.DataFrame({'userid': sampled_userids})
for col in ['gender', 'age', 'country', 'signup']:
    profiles[col] = None  # fill later if TSV becomes available

profiles.to_csv('../data/processed/profiles.csv', index=False)
print(f'Stub profiles.csv saved: {len(profiles)} users')

## 4. Fetch scrobbles via Last.fm API

Results are saved incrementally per user to `data/processed/lastfm_user_cache/<username>.csv`.
Re-running this cell resumes from where it left off.

In [ ]:
from src.data.lastfm_client import fetch_all_users

# ── This cell makes real API calls — safe to re-run (resumes from cache). ───
scrobbles = fetch_all_users(
    network=network,
    userids=sampled_userids,
    lookback_years=dc['lookback_years'],
    page_size=dc['lastfm_page_size'],
    request_delay=dc['lastfm_request_delay'],
    min_scrobbles=dc['min_scrobbles_threshold'],
    save_dir='../data/processed/lastfm_user_cache',
    resume=True,
)

print(f'Scrobbles fetched : {len(scrobbles):,} rows')
print(f'Active users      : {scrobbles["userid"].nunique():,}')
print(f'Date range        : {scrobbles["timestamp"].min()} → {scrobbles["timestamp"].max()}')

In [ ]:
# Deduplicate and save as CSV
scrobbles = scrobbles.drop_duplicates(
    subset=['userid', 'timestamp', 'artist_name', 'track_name']
).sort_values(['userid', 'timestamp']).reset_index(drop=True)

scrobbles.to_csv('../data/processed/scrobbles_updated.csv', index=False)
print(f'Saved scrobbles_updated.csv: {len(scrobbles):,} rows, {scrobbles["userid"].nunique():,} users')
scrobbles.head()

In [ ]:
# Quick overview of scrobble volume distribution
import matplotlib.pyplot as plt

plays_per_user = scrobbles.groupby('userid').size()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

plays_per_user.hist(ax=axes[0], bins=50)
axes[0].set_title('Scrobbles per User')
axes[0].set_xlabel('Scrobble count')

scrobbles['timestamp'] = scrobbles['timestamp'].astype('datetime64[ns, UTC]', errors='ignore')
scrobbles['year'] = scrobbles['timestamp'].astype(str).str[:4]
scrobbles['year'].value_counts().sort_index().plot(kind='bar', ax=axes[1])
axes[1].set_title('Scrobbles by Year')

plt.tight_layout()
plt.savefig('../outputs/figures/scrobble_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Median scrobbles/user: {plays_per_user.median():.0f}')
print(f'Max scrobbles/user  : {plays_per_user.max():,}')

## 5. Enrich with Spotify genres

In [ ]:
import json
from src.data.spotify_client import build_client, fetch_artist_genres

sp = build_client()
unique_artists = scrobbles['artist_name'].dropna().unique().tolist()
print(f'Unique artists to look up: {len(unique_artists):,}')

In [ ]:
# Fetches genres with persistent CSV cache — safe to re-run.
# The 'genres' column is a Python list[str] in memory; serialised as a
# JSON string in the CSV so it can be faithfully round-tripped.
artist_genres = fetch_artist_genres(
    sp,
    unique_artists,
    cache_path='../data/processed/spotify_artist_cache.csv',
    request_delay=0.1,
)

# Serialise genres list → JSON string before saving to CSV
df_genres_out = artist_genres.copy()
df_genres_out['genres'] = df_genres_out['genres'].apply(json.dumps)
df_genres_out.to_csv('../data/processed/artist_genres.csv', index=False)

found = artist_genres['spotify_artist_id'].notna().sum()
print(f'Artists matched on Spotify: {found:,} / {len(artist_genres):,}')
artist_genres.head()

## 6. Enrich with Spotify audio features

In [ ]:
from src.data.spotify_client import fetch_audio_features

# Sample top 20 tracks per user to keep API calls manageable
TOP_TRACKS_PER_USER = 20
track_sample = (
    scrobbles.groupby(['userid', 'artist_name', 'track_name'])
    .size().reset_index(name='play_count')
    .sort_values(['userid', 'play_count'], ascending=[True, False])
    .groupby('userid').head(TOP_TRACKS_PER_USER)
)
track_pairs = list(set(zip(track_sample['artist_name'], track_sample['track_name'])))
print(f'Unique (artist, track) pairs to enrich: {len(track_pairs):,}')

In [ ]:
# Fetches audio features with persistent CSV cache — safe to re-run.
audio_features = fetch_audio_features(
    sp,
    track_pairs,
    cache_path='../data/processed/spotify_audio_features_cache.csv',
    request_delay=0.1,
)

audio_features.to_csv('../data/processed/audio_features.csv', index=False)
print(f'Audio features fetched: {len(audio_features):,} tracks')
audio_features.head()

## Summary

All enriched data is saved to `data/processed/` as CSV files:

| File | Contents |
|---|---|
| `scrobbles_updated.csv` | Scrobble history for ~10,000 sampled users |
| `profiles.csv` | User stub (populate from TSV if available) |
| `artist_genres.csv` | Spotify genre tags (`genres` column is JSON-serialised) |
| `audio_features.csv` | Spotify audio features for top tracks |

Proceed to **Notebook 02** for feature engineering.

> **Tip — loading `artist_genres.csv` in later notebooks:**
> ```python
> from src.data.loader import load_artist_genres_csv
> artist_genres = load_artist_genres_csv('../data/processed/artist_genres.csv')
> # genres column is now a proper Python list[str]
> ```

## 7. EDA — Raw Data Health Check

Quick descriptive statistics on all collected files using **DuckDB** (queries run directly on the CSV files — no full load into RAM required).

In [ ]:
import duckdb

# ── Scrobbles overview ───────────────────────────────────────────────
print('=== SCROBBLES ===')
duckdb.sql("""
    SELECT
        COUNT(*)                                AS total_rows,
        COUNT(DISTINCT userid)                  AS unique_users,
        COUNT(DISTINCT artist_name)             AS unique_artists,
        COUNT(DISTINCT track_name)              AS unique_tracks,
        MIN(timestamp)                          AS earliest_play,
        MAX(timestamp)                          AS latest_play,
        ROUND(COUNT(*) * 1.0
              / COUNT(DISTINCT userid), 1)     AS avg_scrobbles_per_user,
        SUM(CASE WHEN artist_name = '' OR
                       artist_name IS NULL THEN 1 ELSE 0 END) AS missing_artist,
        SUM(CASE WHEN track_name  = '' OR
                       track_name  IS NULL THEN 1 ELSE 0 END) AS missing_track
    FROM read_csv_auto('../data/processed/scrobbles_updated.csv',
                        header=True)
""").show()


In [ ]:
# ── Scrobbles per user distribution ─────────────────────────────────
print('=== SCROBBLES PER USER — percentile distribution ===')
duckdb.sql("""
    WITH counts AS (
        SELECT userid, COUNT(*) AS n
        FROM read_csv_auto('../data/processed/scrobbles_updated.csv', header=True)
        GROUP BY userid
    )
    SELECT
        MIN(n)                         AS min_scrobbles,
        PERCENTILE_CONT(0.10) WITHIN GROUP (ORDER BY n) AS p10,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY n) AS p25,
        PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY n) AS median,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY n) AS p75,
        PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY n) AS p90,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY n) AS p95,
        MAX(n)                         AS max_scrobbles,
        ROUND(AVG(n), 1)               AS mean_scrobbles
    FROM counts
""").show()


In [ ]:
# ── Top 15 artists across all users ─────────────────────────────────
print('=== TOP 15 ARTISTS (by total play count) ===')
duckdb.sql("""
    SELECT artist_name,
           COUNT(*)                   AS total_plays,
           COUNT(DISTINCT userid)      AS listener_count
    FROM read_csv_auto('../data/processed/scrobbles_updated.csv', header=True)
    GROUP BY artist_name
    ORDER BY total_plays DESC
    LIMIT 15
""").show()


In [ ]:
# ── Top 15 tracks across all users ──────────────────────────────────
print('=== TOP 15 TRACKS (by total play count) ===')
duckdb.sql("""
    SELECT artist_name, track_name,
           COUNT(*)              AS total_plays,
           COUNT(DISTINCT userid) AS listener_count
    FROM read_csv_auto('../data/processed/scrobbles_updated.csv', header=True)
    GROUP BY artist_name, track_name
    ORDER BY total_plays DESC
    LIMIT 15
""").show()


In [ ]:
# ── Artist genres overview ───────────────────────────────────────────
print('=== ARTIST GENRES ===')
duckdb.sql("""
    SELECT
        COUNT(*)                                        AS total_artists,
        SUM(CASE WHEN spotify_artist_id IS NOT NULL
                  THEN 1 ELSE 0 END)                   AS matched_on_spotify,
        ROUND(AVG(popularity), 1)                       AS avg_popularity
    FROM read_csv_auto('../data/processed/artist_genres.csv', header=True)
""").show()

# Show null rate per column
import pandas as pd
ag = pd.read_csv('../data/processed/artist_genres.csv')
print('\nNull / empty rates:')
print(ag.isnull().mean().round(3).to_string())


In [ ]:
# ── Audio features overview ──────────────────────────────────────────
print('=== AUDIO FEATURES ===')
audio_cols = [
    'danceability', 'energy', 'valence', 'tempo',
    'acousticness', 'instrumentalness', 'liveness', 'speechiness'
]
af = pd.read_csv('../data/processed/audio_features.csv')
print(f'Rows: {len(af):,}  |  Resolved tracks: {af["track_spotify_id"].notna().sum():,}')
print()
print(af[audio_cols].describe().round(3).to_string())


In [ ]:
# ── Profiles overview ────────────────────────────────────────────────
print('=== PROFILES ===')
duckdb.sql("""
    SELECT
        COUNT(*)                     AS total_users,
        COUNT(DISTINCT country)      AS unique_countries,
        ROUND(AVG(TRY_CAST(age AS DOUBLE)), 1) AS avg_age,
        MIN(TRY_CAST(age AS DOUBLE)) AS min_age,
        MAX(TRY_CAST(age AS DOUBLE)) AS max_age
    FROM read_csv_auto('../data/processed/profiles.csv', header=True)
""").show()

print('\nGender distribution:')
duckdb.sql("""
    SELECT gender, COUNT(*) AS n,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM read_csv_auto('../data/processed/profiles.csv', header=True)
    GROUP BY gender ORDER BY n DESC
""").show()

print('\nTop 10 countries:')
duckdb.sql("""
    SELECT country, COUNT(*) AS n
    FROM read_csv_auto('../data/processed/profiles.csv', header=True)
    GROUP BY country ORDER BY n DESC LIMIT 10
""").show()
